In [ ]:
import mysql.connector
import pandas as pd
import json
import nltk
import os
import keyring
from nltk.tokenize import word_tokenize

In [ ]:
# Setup MySQL connection
mydb = mysql.connector.connect(
    host="localhost",
    user="root",
    password="rootr00t",
    database="Spotify_test_2"  # Make sure to specify your database
)
# cursor = mydb.cursor()

def insert_playlist(playlist):
    """
    Inserts a playlist into the `playlists` table.
    
    This function takes a dictionary containing playlist details (such as `pid` and `name`)
    and inserts the data into the `playlists` table of the database. If an entry with the
    same `pid` already exists, it will be ignored due to the `INSERT IGNORE` SQL statement.
    
    Args:
        :param playlist (dict): A dictionary containing keys:
            :param 'pid' (int or str): The playlist ID.
            :param 'name' (str): The name of the playlist.
    
    :return 
        The `pid` (playlist ID) of the inserted playlist.
    
    Notes:
        This function assumes that `cursor` and `mydb` are previously defined objects
        connecting to the database.
    
    """
    insert_playlist_query = """
    INSERT IGNORE INTO playlists (pid, name)
    VALUES (%s, %s)
    """
    playlist_data = (playlist['pid'], playlist['name'])
    cursor.execute(insert_playlist_query, playlist_data)
    mydb.commit()

    return playlist['pid']


# Function to insert song data
def insert_song(track_uri, artist_name, track_name, duration_ms, artist_uri):
    insert_song_query = """
    INSERT INTO songs (track_uri, artist_name, track_name, duration_ms, artist_uri)
    VALUES (%s, %s, %s, %s, %s) 
    ON DUPLICATE KEY UPDATE artist_uri = VALUES(artist_uri)
    """
    cursor.execute(insert_song_query, (track_uri, artist_name, track_name, duration_ms, artist_uri))
    
# Function to insert playlist_song_map data
def insert_playlist_song_map(pid, track_uri):
    insert_mapping_query = """
    INSERT INTO playlist_song_map (pid, track_uri)
    VALUES (%s, %s)
    """
    cursor.execute(insert_mapping_query, (pid, track_uri))

In [ ]:
print(help(insert_playlist()))

In [ ]:
# nltk.download('punkt')

directory_path = './Training Playlists'

for file_name in os.listdir(directory_path):
    if file_name.endswith('.json'):
        file_path = os.path.join(directory_path, file_name)
        with open(file_path, 'r') as file:
            data = json.load(file)
            for playlist in data['playlists']:
                # Insert playlist and retrieve its ID (PID)
                pid = insert_playlist(playlist)

                # Tokenizing the playlist name using NLTK
                tokenized_name = word_tokenize(playlist['name'])  # This uses NLTK's tokenizer
                # Convert the list of tokens back to a string
                tokenized_name_str = ','.join(tokenized_name)  # Join tokens with a comma

                # Update the table with the tokenized name
                update_query = "UPDATE playlists SET token_names = %s WHERE pid = %s"
                cursor.execute(update_query, (tokenized_name_str, pid))

                # Iterate through tracks in the playlist and insert the songs and the map
                for track in playlist['tracks']:
                    # Insert song data
                    insert_song(track['track_uri'], track['artist_name'], track['track_name'], track['duration_ms'])
                    # Insert mapping data
                    insert_playlist_song_map(pid, track['track_uri'])

            # Commit the inserts and updates after handling each file
            mydb.commit()

In [ ]:
directory_path = './Training Playlists'
# Iterate over every file in the directory
for file_name in os.listdir(directory_path):
    if file_name.endswith('.json'):
        file_path = os.path.join(directory_path, file_name)
        # Open the file
        with open(file_path, 'r') as file:
            # Load JSON data from file
            data = json.load(file)
            # Iterate through each playlist in data
            for playlist in data['playlists']:
                # Iterate through each track in the playlist
                for track in playlist['tracks']:
                    # For each track, insert or update song data in 'songs' table
                    insert_song(
                        track['track_uri'],
                        track['artist_name'],
                        track['track_name'],
                        track['duration_ms'],
                        track['artist_uri']  # Assuming 'artist_uri' is part of your track data
                    )
                # Commit the inserts or updates after handling each file
                mydb.commit()

In [ ]:
!pip install mlxtend

In [ ]:

# Use pandas to execute SQL query to fetch data
# This query joins playlist_song_map with songs on track_uri
query = """
SELECT psm.pid, psm.track_uri, s.artist_uri 
FROM playlist_song_map psm
JOIN songs s ON s.track_uri = psm.track_uri
"""
df = pd.read_sql(query, mydb)

# For handling large results, we process each batch separately
batch_size = 1000

for i, df_chunk in enumerate(pd.read_sql(query, mydb, chunksize=batch_size)):

    # Drop duplicates
    df_chunk = df_chunk.drop_duplicates()

    # Prepare for pivot
    df_chunk['value'] = 1
    df_chunk['track_artist_uri'] = df_chunk['track_uri'] + "_" + df_chunk['artist_uri']

    # Perform pivot
    data = df_chunk.pivot(index='pid', columns='track_artist_uri', values='value').fillna(0)

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# Compute frequent itemsets using the Apriori algorithm
frequent_itemsets = apriori(data, min_support=0.01, use_colnames=True)

# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
print(rules)

In [ ]:
# Predict which song fits the best among the given candidates
def predict_best_fit(playlist_id, candidate_songs, rules):
    candidate_scores = {}

    # Add songs in the playlist
    playlist_songs = set(data.loc[playlist_id][data.loc[playlist_id] == 1].index)

    for song in candidate_songs:
        # Find rules where this song is the consequent
        consequent_rules = rules[rules['consequents'].apply(lambda x: song in x)]

        # Filter for rules where the antecedent is a song in the playlist
        applicable_rules = consequent_rules[consequent_rules['antecedents'].apply(lambda x: bool(playlist_songs & set(x)))]

        # Compute the score of this song as the average confidence of the applicable rules
        avg_confidence = applicable_rules['confidence'].mean()

        candidate_scores[song] = avg_confidence

    # Get the song with maximum average confidence
    best_fit_song = max(candidate_scores, key=candidate_scores.get)

    return best_fit_song